# CMIP6 2020–2070 State + County CSV Builder

This notebook prepares data for the DSC 106 final project story:

**Average warming is meaningful, but extreme hot days make the future feel more personal.**

It downloads/loads CMIP6 data and outputs both **county-level** and **state-level** CSV files:

1. `county_annual_tas_cmip6_2020_2070.csv`
2. `state_annual_tas_cmip6_2020_2070.csv`
3. `county_annual_extreme_heat_cmip6_2020_2070.csv`
4. `state_annual_extreme_heat_cmip6_2020_2070.csv`
5. `us_states.geojson`
6. `us_counties.geojson`

Risk index columns can be added later in a separate notebook using these CSVs.

**Important interpretation note:** county-level values are estimated by sampling the nearest CMIP6 grid cell at each county's representative point. They are useful for exploratory spatial comparison, not precise county-scale prediction.

## 0. Install packages

Run this first in Google Colab.

In [ ]:
%%capture
!pip install geopandas pyogrio shapely rtree fsspec gcsfs zarr xarray dask[complete] netCDF4 cftime intake-esm tqdm

## 1. Imports and configuration

The main setup is:

- Baseline year: **2020**
- Projection years: **2021–2070**
- Full downloaded range: **2020–2070**
- Scenarios: **SSP126, SSP245, SSP585**
- Model: **GFDL-ESM4**
- Member: **r1i1p1f1**

In [ ]:
import os
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import intake
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
xr.set_options(display_style="text")

OUTPUT_DIR = Path("/content/cmip6_2020_2070_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------
# Time setup
# -----------------------
BASELINE_YEAR = 2020
START_YEAR = 2020
END_YEAR = 2070
ALL_YEARS = list(range(START_YEAR, END_YEAR + 1))
PROJECTED_YEARS = list(range(BASELINE_YEAR + 1, END_YEAR + 1))

# -----------------------
# CMIP6 setup
# -----------------------
SOURCE_ID = "GFDL-ESM4"
MEMBER_ID = "r1i1p1f1"
SCENARIOS = ["ssp126", "ssp245", "ssp585"]
SCENARIO_LABELS = {
    "ssp126": "Low emissions (SSP126)",
    "ssp245": "Medium emissions (SSP245)",
    "ssp585": "High emissions (SSP585)",
}

# Extreme heat thresholds in Celsius.
HOT_THRESHOLDS_C = [30, 35]

# Census cartographic boundary files.
COUNTY_ZIP_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_20m.zip"
STATE_ZIP_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_20m.zip"

# Keep 50 states only. Exclude DC and territories.
STATE_ABBRS_50 = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS",
    "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY",
    "NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV",
    "WI","WY"
}

print("Output directory:", OUTPUT_DIR)
print("Years:", START_YEAR, "to", END_YEAR)
print("Scenarios:", SCENARIOS)

## 2. Load state and county boundaries

This creates county representative points for sampling CMIP6 grid cells.

In [ ]:
states = gpd.read_file(STATE_ZIP_URL).to_crs("EPSG:4326")
counties = gpd.read_file(COUNTY_ZIP_URL).to_crs("EPSG:4326")

# Keep only 50 states.
states = states[states["STUSPS"].isin(STATE_ABBRS_50)].copy()

# Join state names/abbreviations onto counties.
state_lookup = states[["STATEFP", "NAME", "STUSPS"]].rename(
    columns={"NAME": "state", "STUSPS": "state_abbr"}
)

counties = counties.merge(state_lookup, on="STATEFP", how="inner")
counties = counties[counties["state_abbr"].isin(STATE_ABBRS_50)].copy()
counties = counties.rename(columns={"NAME": "county"})

# Make sure key columns are strings.
counties["GEOID"] = counties["GEOID"].astype(str).str.zfill(5)
counties["COUNTYFP"] = counties["COUNTYFP"].astype(str).str.zfill(3)
counties["STATEFP"] = counties["STATEFP"].astype(str).str.zfill(2)

county_meta = counties[
    ["GEOID", "STATEFP", "COUNTYFP", "county", "state", "state_abbr", "ALAND", "geometry"]
].copy()

# Representative point is safer than centroid because it stays inside the polygon.
rep_points = county_meta.geometry.representative_point()
county_meta["rep_lon"] = rep_points.x
county_meta["rep_lat"] = rep_points.y

county_attrs = county_meta[["GEOID", "county", "state", "state_abbr", "ALAND"]].copy()

print("States:", len(states))
print("Counties:", len(county_meta))
display(county_meta[["GEOID", "county", "state", "state_abbr", "rep_lon", "rep_lat"]].head())

## 3. Open the CMIP6 catalog and check availability

This notebook uses the Pangeo CMIP6 public catalog on Google Cloud.

In [ ]:
CATALOG_URL = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
cat = intake.open_esm_datastore(CATALOG_URL)

print("Catalog rows:", len(cat.df))
display(cat.df.head())

In [ ]:
# Check whether the needed datasets exist.
REQUIRED_DATASETS = [
    ("tas", "Amon"),
    ("tasmax", "day"),
]

availability_rows = []

for scenario in SCENARIOS:
    for variable_id, table_id in REQUIRED_DATASETS:
        search = cat.search(
            source_id=SOURCE_ID,
            experiment_id=scenario,
            table_id=table_id,
            variable_id=variable_id,
            member_id=MEMBER_ID,
        )
        availability_rows.append({
            "scenario": scenario,
            "variable_id": variable_id,
            "table_id": table_id,
            "matches": len(search.df),
        })

availability = pd.DataFrame(availability_rows)
display(availability)

if (availability["matches"] == 0).any():
    raise ValueError("At least one required CMIP6 dataset is missing. Check SOURCE_ID, MEMBER_ID, scenarios, or variables.")

## 4. Helper functions

In [ ]:
def _get_coord_name(ds, possible_names):
    for name in possible_names:
        if name in ds.coords or name in ds.dims:
            return name
    raise ValueError(
        f"Could not find coordinate among {possible_names}. "
        f"Available coords: {list(ds.coords)}"
    )


def open_cmip6_variable(scenario, variable_id, table_id):
    # Open one CMIP6 variable for one scenario and subset to 2020-2070.
    search = cat.search(
        source_id=SOURCE_ID,
        experiment_id=scenario,
        table_id=table_id,
        variable_id=variable_id,
        member_id=MEMBER_ID,
    )

    if len(search.df) == 0:
        raise ValueError(
            f"No CMIP6 catalog entries found for "
            f"scenario={scenario}, variable={variable_id}, table={table_id}"
        )

    print(
        f"Found {len(search.df)} catalog entries for "
        f"{scenario}, {variable_id}, {table_id}. Using first matching dataset."
    )

    dsets = search.to_dataset_dict(
        zarr_kwargs={"consolidated": True, "use_cftime": True},
        storage_options={"token": "anon"},
        progressbar=True,
    )

    ds = list(dsets.values())[0]

    lat_name = _get_coord_name(ds, ["lat", "latitude", "y"])
    lon_name = _get_coord_name(ds, ["lon", "longitude", "x"])

    da = ds[variable_id]
    da = da.sel(time=slice(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"))

    return da, lat_name, lon_name


def to_celsius(da):
    # CMIP6 temperature variables are usually in Kelvin; convert to Celsius.
    return da - 273.15


def sample_county_timeseries(da, lat_name, lon_name, county_meta):
    # Sample an xarray DataArray at county representative points using nearest grid cell.
    lon_values = da[lon_name].values
    lon_min = np.nanmin(lon_values)
    lon_max = np.nanmax(lon_values)

    county_lons = county_meta["rep_lon"].to_numpy()

    # CMIP6 longitudes are often 0..360. Convert county lon if needed.
    if lon_min >= 0 and lon_max > 180:
        county_lons_for_model = (county_lons + 360) % 360
    else:
        county_lons_for_model = county_lons

    county_lats_da = xr.DataArray(
        county_meta["rep_lat"].to_numpy(),
        dims="county_index",
        coords={"county_index": county_meta["GEOID"].to_numpy()},
    )

    county_lons_da = xr.DataArray(
        county_lons_for_model,
        dims="county_index",
        coords={"county_index": county_meta["GEOID"].to_numpy()},
    )

    sampled = da.sel(
        {
            lat_name: county_lats_da,
            lon_name: county_lons_da,
        },
        method="nearest",
    )

    sampled = sampled.assign_coords(county_index=county_meta["GEOID"].to_numpy())
    return sampled


def annual_da_to_county_df(annual_da, value_name, scenario):
    # Convert annual xarray DataArray with dims time and county_index into a clean DataFrame.
    df = annual_da.to_dataframe(name=value_name).reset_index()

    # Keep only useful columns. Some datasets include lat/lon or height coordinates after sampling.
    keep_cols = [c for c in ["county_index", "time", value_name] if c in df.columns]
    df = df[keep_cols].copy()

    df["year"] = df["time"].apply(lambda t: t.year)
    df = df[df["year"].isin(ALL_YEARS)].copy()
    df = df.rename(columns={"county_index": "GEOID"})
    df["GEOID"] = df["GEOID"].astype(str).str.zfill(5)
    df["scenario"] = scenario
    df["scenario_label"] = SCENARIO_LABELS[scenario]

    return df[["GEOID", "year", "scenario", "scenario_label", value_name]]


def weighted_average(group, value_col, weight_col="ALAND"):
    valid = (
        group[value_col].notna() &
        group[weight_col].notna() &
        (group[weight_col] > 0)
    )
    if valid.sum() == 0:
        return np.nan
    return np.average(group.loc[valid, value_col], weights=group.loc[valid, weight_col])


def aggregate_county_to_state(county_df, value_cols):
    # Area-weight county rows to state rows using county ALAND.
    group_cols = ["state", "state_abbr", "year", "scenario", "scenario_label"]
    state_records = []

    for keys, group in county_df.groupby(group_cols, dropna=False):
        record = dict(zip(group_cols, keys))
        for col in value_cols:
            record[col] = weighted_average(group, col)
        state_records.append(record)

    return pd.DataFrame(state_records).sort_values(["state", "scenario", "year"])

## 5. Build county + state annual average temperature CSVs

This uses `tas` from CMIP6 monthly data (`Amon`) and computes annual average temperature in Celsius.

In [ ]:
county_tas_parts = []

for scenario in SCENARIOS:
    print("\n==============================")
    print("Processing annual tas:", scenario)
    print("==============================")

    tas, lat_name, lon_name = open_cmip6_variable(
        scenario=scenario,
        variable_id="tas",
        table_id="Amon",
    )

    sampled_tas = sample_county_timeseries(tas, lat_name, lon_name, county_meta)
    sampled_tas_c = to_celsius(sampled_tas)
    sampled_tas_c.name = "tas_c"

    # Monthly tas -> annual average.
    annual_tas_c = sampled_tas_c.resample(time="YS").mean()

    county_scenario_tas = annual_da_to_county_df(
        annual_da=annual_tas_c,
        value_name="tas_c",
        scenario=scenario,
    )

    county_tas_parts.append(county_scenario_tas)

    # Free memory before next scenario.
    del tas, sampled_tas, sampled_tas_c, annual_tas_c, county_scenario_tas
    gc.collect()

county_tas = pd.concat(county_tas_parts, ignore_index=True)
county_tas = county_tas.merge(county_attrs, on="GEOID", how="left")

# Scenario-specific 2020 model baseline.
baseline_tas = county_tas[county_tas["year"] == BASELINE_YEAR][
    ["GEOID", "scenario", "tas_c"]
].rename(columns={"tas_c": "tas_2020_c"})

county_tas = county_tas.merge(
    baseline_tas,
    on=["GEOID", "scenario"],
    how="left",
)

county_tas["tas_change_from_2020_c"] = county_tas["tas_c"] - county_tas["tas_2020_c"]

county_tas = county_tas[
    [
        "GEOID", "county", "state", "state_abbr", "ALAND",
        "year", "scenario", "scenario_label",
        "tas_c", "tas_2020_c", "tas_change_from_2020_c",
    ]
].sort_values(["state", "county", "scenario", "year"])

state_tas = aggregate_county_to_state(
    county_tas,
    value_cols=["tas_c", "tas_2020_c", "tas_change_from_2020_c"],
)

print("county_tas rows:", len(county_tas))
print("state_tas rows:", len(state_tas))
display(county_tas.head())
display(state_tas.head())

In [ ]:
county_tas_csv = OUTPUT_DIR / "county_annual_tas_cmip6_2020_2070.csv"
state_tas_csv = OUTPUT_DIR / "state_annual_tas_cmip6_2020_2070.csv"

county_tas.to_csv(county_tas_csv, index=False)
state_tas.to_csv(state_tas_csv, index=False)

print("Saved:")
print(county_tas_csv)
print(state_tas_csv)

## 6. Build county + state annual extreme heat CSVs

This uses `tasmax` from CMIP6 daily data (`day`) and computes:

- annual maximum daily max temperature
- number of days above 30°C
- number of days above 35°C
- changes in hot-day counts from the 2020 model baseline

This cell can take longer than the temperature cell because `tasmax` is daily data.

In [ ]:
county_heat_parts = []

for scenario in SCENARIOS:
    print("\n==============================")
    print("Processing daily tasmax:", scenario)
    print("==============================")

    tasmax, lat_name, lon_name = open_cmip6_variable(
        scenario=scenario,
        variable_id="tasmax",
        table_id="day",
    )

    sampled_tasmax = sample_county_timeseries(tasmax, lat_name, lon_name, county_meta)
    sampled_tasmax_c = to_celsius(sampled_tasmax)
    sampled_tasmax_c.name = "tasmax_c"

    # Annual max of daily maximum temperature.
    annual_max_tasmax_c = sampled_tasmax_c.resample(time="YS").max()
    annual_max_df = annual_da_to_county_df(
        annual_da=annual_max_tasmax_c,
        value_name="annual_max_tasmax_c",
        scenario=scenario,
    )

    # Annual hot day counts.
    hot_count_dfs = []
    for threshold in HOT_THRESHOLDS_C:
        value_name = f"hot_days_{threshold}c"
        print(f"  Counting days above {threshold}°C")
        hot_days = (sampled_tasmax_c > threshold).resample(time="YS").sum()
        hot_days.name = value_name

        hot_df = annual_da_to_county_df(
            annual_da=hot_days,
            value_name=value_name,
            scenario=scenario,
        )
        hot_count_dfs.append(hot_df)

    combined = annual_max_df.copy()
    for hot_df in hot_count_dfs:
        combined = combined.merge(
            hot_df,
            on=["GEOID", "year", "scenario", "scenario_label"],
            how="left",
        )

    county_heat_parts.append(combined)

    # Free memory before next scenario.
    del tasmax, sampled_tasmax, sampled_tasmax_c, annual_max_tasmax_c, annual_max_df
    del hot_count_dfs, combined
    gc.collect()

county_heat = pd.concat(county_heat_parts, ignore_index=True)
county_heat = county_heat.merge(county_attrs, on="GEOID", how="left")

# Scenario-specific 2020 model baseline for hot days.
baseline_heat_cols = ["GEOID", "scenario"] + [f"hot_days_{t}c" for t in HOT_THRESHOLDS_C]
baseline_heat = county_heat[county_heat["year"] == BASELINE_YEAR][baseline_heat_cols].copy()
baseline_heat = baseline_heat.rename(columns={
    f"hot_days_{t}c": f"hot_days_{t}c_2020" for t in HOT_THRESHOLDS_C
})

county_heat = county_heat.merge(
    baseline_heat,
    on=["GEOID", "scenario"],
    how="left",
)

for threshold in HOT_THRESHOLDS_C:
    county_heat[f"hot_days_{threshold}c_change_from_2020"] = (
        county_heat[f"hot_days_{threshold}c"] - county_heat[f"hot_days_{threshold}c_2020"]
    )

heat_cols = [
    "GEOID", "county", "state", "state_abbr", "ALAND",
    "year", "scenario", "scenario_label",
    "annual_max_tasmax_c",
]

for threshold in HOT_THRESHOLDS_C:
    heat_cols += [
        f"hot_days_{threshold}c",
        f"hot_days_{threshold}c_2020",
        f"hot_days_{threshold}c_change_from_2020",
    ]

county_heat = county_heat[heat_cols].sort_values(["state", "county", "scenario", "year"])

state_heat_value_cols = ["annual_max_tasmax_c"]
for threshold in HOT_THRESHOLDS_C:
    state_heat_value_cols += [
        f"hot_days_{threshold}c",
        f"hot_days_{threshold}c_2020",
        f"hot_days_{threshold}c_change_from_2020",
    ]

state_heat = aggregate_county_to_state(
    county_heat,
    value_cols=state_heat_value_cols,
)

print("county_heat rows:", len(county_heat))
print("state_heat rows:", len(state_heat))
display(county_heat.head())
display(state_heat.head())

In [ ]:
county_heat_csv = OUTPUT_DIR / "county_annual_extreme_heat_cmip6_2020_2070.csv"
state_heat_csv = OUTPUT_DIR / "state_annual_extreme_heat_cmip6_2020_2070.csv"

county_heat.to_csv(county_heat_csv, index=False)
state_heat.to_csv(state_heat_csv, index=False)

print("Saved:")
print(county_heat_csv)
print(state_heat_csv)

## 7. Export GeoJSON files for D3

In [ ]:
states_export = states[["STATEFP", "STUSPS", "NAME", "geometry"]].copy()
states_export = states_export.rename(columns={
    "STUSPS": "state_abbr",
    "NAME": "state",
})

counties_export = county_meta[
    ["GEOID", "STATEFP", "COUNTYFP", "county", "state", "state_abbr", "geometry"]
].copy()

states_geojson = OUTPUT_DIR / "us_states.geojson"
counties_geojson = OUTPUT_DIR / "us_counties.geojson"

states_export.to_file(states_geojson, driver="GeoJSON")
counties_export.to_file(counties_geojson, driver="GeoJSON")

print("Saved:")
print(states_geojson)
print(counties_geojson)

## 8. Quick validation checks

These checks help catch obvious problems before downloading.

In [ ]:
print("Files in output directory:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name, "--", round(path.stat().st_size / 1_000_000, 2), "MB")

print("\nMissing values check:")
for name, df in [
    ("county_tas", county_tas),
    ("state_tas", state_tas),
    ("county_heat", county_heat),
    ("state_heat", state_heat),
]:
    print("\n", name)
    print(df.isna().sum().sort_values(ascending=False).head(10))

print("\nYear ranges:")
for name, df in [
    ("county_tas", county_tas),
    ("state_tas", state_tas),
    ("county_heat", county_heat),
    ("state_heat", state_heat),
]:
    print(name, df["year"].min(), df["year"].max(), "rows:", len(df))

## 9. Zip and download all outputs

In [ ]:
import shutil

zip_path = shutil.make_archive(
    base_name=str(OUTPUT_DIR),
    format="zip",
    root_dir=str(OUTPUT_DIR),
)

print("Created zip:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("If this is not running in Colab, download manually from:", zip_path)
    print(e)

## 10. Optional: merge temperature + heat into one county/state file

You do **not** have to use this for D3, but it can be convenient for later risk-index building.

In [ ]:
county_combined = county_tas.merge(
    county_heat.drop(columns=["county", "state", "state_abbr", "ALAND"]),
    on=["GEOID", "year", "scenario", "scenario_label"],
    how="left",
)

state_combined = state_tas.merge(
    state_heat,
    on=["state", "state_abbr", "year", "scenario", "scenario_label"],
    how="left",
)

county_combined_csv = OUTPUT_DIR / "county_combined_tas_heat_cmip6_2020_2070.csv"
state_combined_csv = OUTPUT_DIR / "state_combined_tas_heat_cmip6_2020_2070.csv"

county_combined.to_csv(county_combined_csv, index=False)
state_combined.to_csv(state_combined_csv, index=False)

print("Saved optional combined files:")
print(county_combined_csv)
print(state_combined_csv)

display(county_combined.head())
display(state_combined.head())